CLASSIFIKASI

In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import json

In [5]:
def get_target_column(info_json_path):
    """
    Ambil nama kolom target dari file info.json.
    Menangani beberapa kemungkinan format:
      1. {"target_col": "income"}                         -> langsung nama kolom
      2. {"target_col_idx": [14], "column_names": [...]}  -> index -> mapping ke nama
      3. {"target_col_idx": [14]} tanpa column_names       -> dikembalikan sebagai index (int)
    """
    with open(info_json_path, "r") as f:
        info = json.load(f)

    if "target_col" in info:
        return info["target_col"]

    if "target_col_idx" in info:
        idx_list = info["target_col_idx"]
        idx = idx_list[0] if isinstance(idx_list, list) else idx_list

        if "column_names" in info:
            return info["column_names"][idx]
        else:
            # fallback: kembalikan index kolom (integer), nanti ditangani saat load CSV
            return idx

    raise ValueError(
        f"Tidak menemukan 'target_col' atau 'target_col_idx' di {info_json_path}. "
        f"Key yang tersedia: {list(info.keys())}"
    )


def load_and_prepare(csv_path, target_col):
    df = pd.read_csv(csv_path)

    # jika target_col berupa index (int), ubah ke nama kolom berdasarkan posisi
    if isinstance(target_col, int):
        target_col = df.columns[target_col]

    if target_col not in df.columns:
        raise ValueError(
            f"Kolom target '{target_col}' tidak ditemukan di {csv_path}. "
            f"Kolom tersedia: {list(df.columns)}"
        )

    y = df[target_col]
    X = df.drop(columns=[target_col])

    for col in X.select_dtypes(include=["object", "category"]).columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

    if y.dtype == object or str(y.dtype) == "category":
        y = LabelEncoder().fit_transform(y)

    return X, y

In [18]:
BASE_DIR = r"D:\KULIAH\RISET\PROGRAM\RESULT"
INFO_DIR = os.path.join(BASE_DIR, "datasets", "info")  # sesuaikan kalau root-nya beda

DATASETS = {
    "adult": {
        "path": os.path.join(BASE_DIR, "ADULT", "MCAR", "60", "CSV"),
        "info_json": os.path.join(INFO_DIR, "adult.json"),
    },
    "shoppers": {
        "path": os.path.join(BASE_DIR, "SHOPPERS", "MCAR", "60", "CSV"),
        "info_json": os.path.join(INFO_DIR, "shoppers.json"),
    },
}

# metode imputasi & pola nama file + jumlah repetisi masing-masing
METHODS = {
    "diffputer": {"pattern": "test_impute_{}.csv",      "n_rep": 2},  # proposed method
    "mrmd":      {"pattern": "test_impute_mrmd_{}.csv", "n_rep": 2},   # baseline
    # tambah baseline lain kalau ada, misal:
    # "mean":  {"pattern": "test_impute_mean_{}.csv", "n_rep": 10},
    # "knn":   {"pattern": "test_impute_knn_{}.csv",  "n_rep": 10},
}

N_SPLITS_CV = 5
RANDOM_STATE = 42

In [9]:
CLASSIFIERS = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True, random_state=RANDOM_STATE)),
    ]),
    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    ),
}

SCORING = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
    "roc_auc": "roc_auc",        # ganti ke "roc_auc_ovr" kalau targetnya multiclass
}

In [20]:
# ============================================================
# FUNGSI BANTU
# ============================================================

def load_and_prepare(csv_path, target_col):
    df = pd.read_csv(csv_path)

    if isinstance(target_col, int):
        target_col = df.columns[target_col]

    if target_col not in df.columns:
        raise ValueError(
            f"Kolom target '{target_col}' tidak ditemukan di {csv_path}. "
            f"Kolom tersedia: {list(df.columns)}"
        )

    y = df[target_col]
    X = df.drop(columns=[target_col])

    # encode kolom kategorikal di X (kalau ada)
    for col in X.select_dtypes(include=["object", "category"]).columns:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

    # bersihkan whitespace di target (mis. " <=50K" -> "<=50K") lalu selalu encode ke angka
    if y.dtype == object or str(y.dtype) == "category":
        y = y.astype(str).str.strip()
    y = LabelEncoder().fit_transform(y)

    return X, y


def evaluate_one_file(X, y, classifiers, scoring, n_splits, random_state):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    rows = []
    for clf_name, clf in classifiers.items():
        scores = cross_validate(clf, X, y, cv=cv, scoring=scoring, n_jobs=-1)
        row = {"classifier": clf_name}
        for metric in scoring:
            key = f"test_{metric}"
            row[f"{metric}_mean"] = np.mean(scores[key])
            row[f"{metric}_std"] = np.std(scores[key])
        rows.append(row)
    return rows

In [21]:
all_results = []

for dataset_name, dataset_cfg in DATASETS.items():
    dataset_path = dataset_cfg["path"]
    target_col = get_target_column(dataset_cfg["info_json"])

    for method_name, method_cfg in METHODS.items():
        pattern = method_cfg["pattern"]
        n_rep = method_cfg["n_rep"]

        for rep in range(n_rep):
            file_path = os.path.join(dataset_path, pattern.format(rep))
            if not os.path.exists(file_path):
                print(f"[SKIP] Tidak ditemukan: {file_path}")
                continue

            print(f"[RUN] dataset={dataset_name} | method={method_name} | rep={rep}")
            X, y = load_and_prepare(file_path, target_col)
            rep_results = evaluate_one_file(
                X, y, CLASSIFIERS, SCORING, N_SPLITS_CV, RANDOM_STATE
            )
            for r in rep_results:
                r.update({
                    "dataset": dataset_name,
                    "method": method_name,
                    "repetition": rep,
                })
                all_results.append(r)

results_df = pd.DataFrame(all_results)

[RUN] dataset=adult | method=diffputer | rep=0


C:\Users\RETNO\AppData\Local\Temp\ipykernel_27572\1222859324.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category"]).columns:


[RUN] dataset=adult | method=diffputer | rep=1


C:\Users\RETNO\AppData\Local\Temp\ipykernel_27572\1222859324.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category"]).columns:


[RUN] dataset=adult | method=mrmd | rep=0


...

C:\Users\RETNO\AppData\Local\Temp\ipykernel_27572\1222859324.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category"]).columns:


[RUN] dataset=shoppers | method=mrmd | rep=0


C:\Users\RETNO\AppData\Local\Temp\ipykernel_27572\1222859324.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category"]).columns:


[RUN] dataset=shoppers | method=mrmd | rep=1


C:\Users\RETNO\AppData\Local\Temp\ipykernel_27572\1222859324.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include=["object", "category"]).columns:


In [23]:
# ============================================================
# RINGKASAN: rata-rata & std ANTAR REPETISI, per dataset-method-classifier
# ============================================================

# tampilkan semua kolom & baris tanpa terpotong
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

# ambil semua metric (accuracy, precision, recall, f1, roc_auc) -> mean & std masing2
metric_cols = [c for c in results_df.columns if c.endswith("_mean") or c.endswith("_std")]

# rata-ratakan hasil antar repetisi, per dataset-method-classifier
summary_df = (
    results_df
    .groupby(["dataset", "method", "classifier"])[metric_cols]
    .mean()
    .reset_index()
    .round(4)
)

# urutkan biar enak dibaca: per dataset, per classifier, baru bandingkan method (diffputer vs mrmd)
summary_df = summary_df.sort_values(["dataset", "classifier", "method"]).reset_index(drop=True)

print("\n=== HASIL DETAIL PER REPETISI ===")
print(results_df.round(4).to_string(index=False))

print("\n=== RINGKASAN (rata-rata antar repetisi, per dataset-method-classifier) ===")
print(summary_df.to_string(index=False))


=== HASIL DETAIL PER REPETISI ===
        classifier  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std  dataset    method  repetition
LogisticRegression         0.7966        0.0047          0.7767         0.0072       0.7966      0.0047   0.7667  0.0056        0.7924       0.0089    adult diffputer           0
               SVM         0.7988        0.0059          0.7823         0.0102       0.7988      0.0059   0.7639  0.0064        0.7771       0.0064    adult diffputer           0
           XGBoost         0.7953        0.0044          0.7777         0.0053       0.7953      0.0044   0.7805  0.0050        0.8059       0.0095    adult diffputer           0
LogisticRegression         0.7906        0.0043          0.7676         0.0067       0.7906      0.0043   0.7586  0.0039        0.7905       0.0025    adult diffputer           1
               SVM         0.7985        0.0047          0.7834       